In [ ]:
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "meta-llama/Llama-3.1-8B"
MAX_NEW_TOKENS = 220
TEMPERATURE = 0.0
MAX_TOKENS_PER_CALL = 100
MAX_RETRIES = 2

# Hugging Face auth token must be available in environment variables.
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise EnvironmentError("HF_TOKEN is not set. Add your Hugging Face token to environment variables.")

print("Mode: Hugging Face local inference")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Model:", MODEL_NAME)
print("Chunk size:", MAX_TOKENS_PER_CALL, "| Retries:", MAX_RETRIES)

Mode: Hugging Face local inference
CUDA available: True
GPU: Tesla T4
Model: meta-llama/Llama-3.1-8B
Chunk size: 100 | Retries: 2


In [ ]:
BASE_DIR_DATA = Path('/kaggle/input/datasets/fasihhk/securiti-internship-data')
BASE_DIR_OUTPUT = Path('/kaggle/working')

train_path = BASE_DIR_DATA / 'data' / 'data.json'
test_path = BASE_DIR_DATA / 'data' / 'test_data.json'

df_train = pd.read_json(train_path)
df_test = pd.read_json(test_path)

print('train rows:', len(df_train))
print('test rows:', len(df_test))
df_test.head(2)

train rows: 28516
test rows: 3650


,lang,ner_tags,sequence,tokens
0,en,"[O, O, O, O, O, O, O, O, O, B-PER, I-PER, O]",included future Rage Against the Machine and A...,"[included, future, Rage, Against, the, Machine..."
1,en,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",The city voted 53.5 percent in favor of the ma...,"[The, city, voted, 53.5, percent, in, favor, o..."


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)

model.generation_config.pad_token_id = tokenizer.pad_token_id

generator = pipeline(
    task='text-generation',
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
)

LABEL_SET = ['O', 'B-PER', 'I-PER', 'B-EMAIL', 'I-EMAIL']

# System instructions constrain output format and label vocabulary.
SYSTEM_PROMPT = (
    "You are a token-level NER tagger for two entity types: PER and EMAIL. "
    "Return STRICT JSON ONLY with EXACT schema: {\"items\":[{\"i\":0,\"tag\":\"O\"}]}. "
    "Rules: (1) Output one item for every input token index, in order. "
    "(2) Use only tags: O, B-PER, I-PER, B-EMAIL, I-EMAIL. "
    "(3) Do not output markdown, code fences, or explanations. "
    "(4) If uncertain, use O."
)

def build_user_prompt_from_tokens(tokens):
    indexed_tokens = [{"i": i, "token": str(tok)} for i, tok in enumerate(tokens)]
    return (
        "Input tokens: [{\"i\":0,\"token\":\"John\"},{\"i\":1,\"token\":\"Doe\"},{\"i\":2,\"token\":\"emailed\"},{\"i\":3,\"token\":\"john@x.com\"}]\n"
        "Output JSON: {\"items\":[{\"i\":0,\"tag\":\"B-PER\"},{\"i\":1,\"tag\":\"I-PER\"},{\"i\":2,\"tag\":\"O\"},{\"i\":3,\"tag\":\"B-EMAIL\"}]}\n"
        "EXAMPLE\n"
        "Input tokens: [{\"i\":0,\"token\":\"Hello\"},{\"i\":1,\"token\":\"team\"}]\n"
        "Output JSON: {\"items\":[{\"i\":0,\"tag\":\"O\"},{\"i\":1,\"tag\":\"O\"}]}\n\n" +
        "Now tag this token list. Return JSON only.\n" +
        f"Input tokens: {json.dumps(indexed_tokens, ensure_ascii=False)}\n"
    )

def _extract_json_blob(raw_text):
    # Remove code fences and parse a JSON object from model text.
    cleaned = re.sub(r'```(?:json)?', '', raw_text, flags=re.IGNORECASE).replace('```', '').strip()

    try:
        obj = json.loads(cleaned)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    m = re.search(r'\{[\s\S]*\}', cleaned)
    if not m:
        return {}

    blob = m.group(0)
    try:
        obj = json.loads(blob)
        if isinstance(obj, dict):
            return obj
    except Exception:
        return {}

    return {}

def _sanitize_tag(tag):
    s = str(tag).strip().upper()
    return s if s in LABEL_SET else 'O'

def _fix_bio_sequence(tags):
    # Repair invalid I- tags that do not continue a matching entity span.
    fixed = []
    for tag in tags:
        if tag.startswith('I-'):
            ent = tag[2:]
            if not fixed or fixed[-1] not in {f'B-{ent}', f'I-{ent}'}:
                tag = f'B-{ent}'
        fixed.append(tag)
    return fixed

def _parse_tags_object(obj, n_tokens):
    # Schema: {'items': [{'i': int, 'tag': str}, ...]}.
    tags = ['O'] * n_tokens
    for item in obj['items']:
        if not isinstance(item, dict):
            continue
        i = item.get('i')
        if isinstance(i, int) and 0 <= i < n_tokens:
            tags[i] = _sanitize_tag(item.get('tag', 'O'))
    return tags

def _build_prompt(messages):
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        sys_msg = messages[0]['content']
        usr_msg = messages[1]['content']
        return f"<|system|>\n{sys_msg}\n\n<|user|>\n{usr_msg}\n\n<|assistant|>\n"

def _infer_once(tokens, repair_mode=False):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt_from_tokens(tokens)},
    ]
    if repair_mode:
        messages[1]['content'] += (
            "\nIMPORTANT: previous output was invalid. Return ONLY JSON with EXACTLY one item per token index."
        )

    prompt = _build_prompt(messages)
    dynamic_max_new_tokens = max(MAX_NEW_TOKENS, min(1200, 40 + len(tokens) * 8))

    out = generator(
        prompt,
        max_new_tokens=dynamic_max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        do_sample=False if TEMPERATURE == 0.0 else True,
        temperature=TEMPERATURE,
    )[0]['generated_text']

    obj = _extract_json_blob(out)
    pred_tags = _parse_tags_object(obj, len(tokens))
    pred_tags = _fix_bio_sequence(pred_tags)

    # All-O responses are retried once to reduce missed-entity failures.
    all_o = all(t == 'O' for t in pred_tags)
    return pred_tags, out, all_o

def run_zeroshot_token_tagger(tokens):
    # Chunk long sequences to keep generation reliable and bounded.
    if len(tokens) > MAX_TOKENS_PER_CALL:
        all_tags = []
        raw_outputs = []
        for start in range(0, len(tokens), MAX_TOKENS_PER_CALL):
            chunk = tokens[start:start + MAX_TOKENS_PER_CALL]
            tags_chunk, raw_chunk = run_zeroshot_token_tagger(chunk)
            all_tags.extend(tags_chunk)
            raw_outputs.append(raw_chunk)
        return all_tags, "\n---CHUNK---\n".join(raw_outputs)

    pred_tags, out, all_o = _infer_once(tokens, repair_mode=False)
    retries = 0
    while retries < MAX_RETRIES and all_o:
        pred_tags, out, all_o = _infer_once(tokens, repair_mode=True)
        retries += 1

    return pred_tags, out

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [ ]:
def normalize_tag_list(tags):
    # Normalize labels that may appear as numeric ids or unexpected strings.
    label_list = ['O', 'B-PER', 'I-PER', 'B-EMAIL', 'I-EMAIL']
    id2label = {i: label for i, label in enumerate(label_list)}

    out = []
    for t in tags:
        s = str(t).strip()
        if s.isdigit():
            s = id2label.get(int(s), 'O')
        out.append(s if s in label_list else 'O')
    return out

def _norm_token(t):
    return re.sub(r'[^a-z0-9@._+-]', '', str(t).lower())

def _find_subsequence(tokens, phrase_tokens):
    if not phrase_tokens or len(phrase_tokens) > len(tokens):
        return None
    for i in range(0, len(tokens) - len(phrase_tokens) + 1):
        if tokens[i:i + len(phrase_tokens)] == phrase_tokens:
            return i, i + len(phrase_tokens)
    return None

def entities_to_token_tags(tokens, entities):
    # Converts span entities into BIO tags.
    pred_tags = ['O'] * len(tokens)
    norm_tokens = [_norm_token(t) for t in tokens]

    ordered = sorted(entities, key=lambda x: 0 if x['type'] == 'EMAIL' else 1)

    for ent in ordered:
        ent_type = ent['type']
        ent_text = ent['text']
        phrase_raw = [p for p in str(ent_text).split() if p.strip()]
        phrase = [_norm_token(p) for p in phrase_raw if _norm_token(p)]

        if not phrase:
            continue

        span = _find_subsequence(norm_tokens, phrase)
        if span is None:
            continue

        s, e = span
        if ent_type == 'PER':
            pred_tags[s] = 'B-PER'
            for i in range(s + 1, e):
                pred_tags[i] = 'I-PER'
        elif ent_type == 'EMAIL':
            pred_tags[s] = 'B-EMAIL'
            for i in range(s + 1, e):
                pred_tags[i] = 'I-EMAIL'

    return pred_tags

def entity_binary_metrics(y_true_labels, y_pred_labels, positive_prefix):
    # B-/I- variants are merged per entity family.
    y_true_bin = [1 if label.endswith(positive_prefix) else 0 for label in y_true_labels]
    y_pred_bin = [1 if label.endswith(positive_prefix) else 0 for label in y_pred_labels]

    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1]).ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_bin, y_pred_bin, average='binary', zero_division=0
    )
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    fnr = fn / (fn + tp) if (fn + tp) else 0.0

    return {
        'accuracy': accuracy,
        'FPR': fpr,
        'FNR': fnr,
        'precision': precision,
        'recall': recall,
        'F1': f1,
        'support_positive': int(sum(y_true_bin)),
    }

In [ ]:
# Evaluate a small subset for faster iteration while testing prompt behavior.
MAX_SAMPLES = 20
eval_df = df_test.head(MAX_SAMPLES).copy()

all_true = []
all_pred = []
records = []

for idx, row in eval_df.iterrows():
    tokens = list(row['tokens'])
    true_tags = normalize_tag_list(row['ner_tags'])

    pred_tags, raw_response = run_zeroshot_token_tagger(tokens)

    all_true.extend(true_tags)
    all_pred.extend(pred_tags)

    # Keep a few examples for inspecting model output formatting.
    if idx < 5:
        records.append({
            'tokens': tokens[:20],
            'true_tags': true_tags[:20],
            'pred_tags': pred_tags[:20],
            'raw_response': raw_response
        })

name_metrics = entity_binary_metrics(all_true, all_pred, 'PER')
email_metrics = entity_binary_metrics(all_true, all_pred, 'EMAIL')

report_df = pd.DataFrame([name_metrics, email_metrics], index=['NAME', 'EMAIL'])
display(report_df)

print(f'Rows evaluated: {len(eval_df)}')
print(f'Tokens evaluated: {len(all_true)}')

Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=544) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

,accuracy,FPR,FNR,precision,recall,F1,support_positive
NAME,0.915743,0.005,0.705882,0.882353,0.294118,0.441176,51
EMAIL,1.000000,0.000,0.000000,0.000000,0.000000,0.000000,0


Rows evaluated: 20
Tokens evaluated: 451


In [ ]:
# Preview sampled records to validate tagging behavior and JSON compliance.
pd.DataFrame(records)

,tokens,true_tags,pred_tags,raw_response
0,"[included, future, Rage, Against, the, Machine...","[O, O, O, O, O, O, O, O, O, B-PER, I-PER, O]","[O, O, O, O, O, O, O, O, O, O, O, O]","{""items"":[{""i"":0,""tag"":""O""},{""i"":1,""tag"":""O""},..."
1,"[The, city, voted, 53.5, percent, in, favor, o...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","{""items"":[{""i"":0,""tag"":""O""},{""i"":1,""tag"":""O""},..."
2,"[It, was, not, until, about, 1907, –, 1909, th...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","{""items"":[{""i"":0,""tag"":""O""},{""i"":1,""tag"":""O""},..."
3,"[Always, in, his, bowler, hat, ,, he, was, a, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","{""items"":[{""i"":0,""tag"":""O""},{""i"":1,""tag"":""O""},..."
4,"[The, music, was, composed, and, conducted, by...","[O, O, O, O, O, O, O, B-PER, I-PER, O]","[O, O, O, O, O, O, O, B-PER, I-PER, O]","{""items"":[{""i"":0,""tag"":""O""},{""i"":1,""tag"":""O""},..."


In [ ]:
# Persist the validation metrics table for reporting.
out_dir = BASE_DIR_OUTPUT / 'reports'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'llm_zero_shot_report.csv'
report_df.to_csv(out_path)
print('Saved:', out_path)

Saved: /kaggle/working/reports/llm_zero_shot_report.csv


## Notes
- This notebook uses prompting only, without task-specific fine-tuning.
- Inference is executed through Hugging Face model loading and generation.
- Output parsing enforces JSON structure and valid BIO tags.
- Long token sequences are processed in chunks, with retry logic for unstable outputs.
- Runtime can be reduced by lowering `MAX_SAMPLES` and `MAX_AI4_SAMPLES`.

## AI4Privacy Zero-Shot Test
This section evaluates the same zero-shot pipeline on `ai4privacy/pii-masking-200k` for an out-of-domain check.

Flow in this section:
- map AI4Privacy labels into `PER` and `EMAIL`
- sample a small subset with entity coverage
- run the same LLM token tagger
- compute and save the metrics report

In [ ]:
CUSTOM_TEXT = "Hi, I am John F. Doe. You can reach me at john.doe@example.com and cc jane.smith@company.org."

import re

# Regex keeps email tokens intact while splitting other text into token-like units.
custom_tokens = re.findall(
    r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|[A-Za-z]+(?:'[A-Za-z]+)?|\d+|[^\w\s]",
    CUSTOM_TEXT,
)

pred_tags, raw_response = run_zeroshot_token_tagger(custom_tokens)

def bio_to_entities(tokens, tags):
    # Converts BIO tags back to contiguous entity spans for readability.
    entities = []
    i = 0
    while i < len(tokens):
        tag = tags[i]
        if tag.startswith("B-"):
            ent_type = tag[2:]
            start = i
            i += 1
            while i < len(tags) and tags[i] == f"I-{ent_type}":
                i += 1
            end = i
            entities.append(
                {
                    "type": ent_type,
                    "text": " ".join(tokens[start:end]),
                    "start_token": start,
                    "end_token_exclusive": end,
                }
            )
        else:
            i += 1
    return entities

detected = bio_to_entities(custom_tokens, pred_tags)

print("Input:", CUSTOM_TEXT)
print("\nTokens:", custom_tokens)
print("\nPredicted tags:", pred_tags)

print("\nDetected entities:")
if detected:
    display(pd.DataFrame(detected))
else:
    print("No PER/EMAIL entities detected.")

Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input: Hi, I am John F. Doe. You can reach me at john.doe@example.com and cc jane.smith@company.org.

Tokens: ['Hi', ',', 'I', 'am', 'John', 'F', '.', 'Doe', '.', 'You', 'can', 'reach', 'me', 'at', 'john.doe@example.com', 'and', 'cc', 'jane.smith@company.org', '.']

Predicted tags: ['O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-EMAIL', 'O', 'O', 'B-EMAIL', 'O']

Detected entities:


,type,text,start_token,end_token_exclusive
0,PER,John F,4,6
1,EMAIL,john.doe@example.com,14,15
2,EMAIL,jane.smith@company.org,17,18


In [ ]:
from datasets import load_dataset

def map_ai4privacy_label(label):
    # Map external labels into the notebook's PER/EMAIL BIO schema.
    s = str(label).upper().strip()
    if s == 'O':
        return 'O'

    prefix = 'B-' if s.startswith('B-') else 'I-' if s.startswith('I-') else None
    base = s[2:] if prefix else s

    if 'EMAIL' in base:
        return (prefix or 'B-') + 'EMAIL'

    person_keys = ['NAME', 'FIRSTNAME', 'LASTNAME', 'FULLNAME', 'MIDDLENAME', 'SURNAME', 'PERSON']
    if any(k in base for k in person_keys):
        return (prefix or 'B-') + 'PER'

    return 'O'

def parse_list_field(value):
    # Handle both native Python lists and list-like strings.
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        value = value.strip()
        if value.startswith('[') and value.endswith(']'):
            try:
                return json.loads(value.replace("'", '"'))
            except Exception:
                try:
                    import ast
                    parsed = ast.literal_eval(value)
                    return parsed if isinstance(parsed, list) else []
                except Exception:
                    return []
    return []

def convert_ai4privacy_example(example):
    # Preferred fields from AI4Privacy mBERT exports.
    tokens = parse_list_field(example.get('mbert_text_tokens', []))
    labels = parse_list_field(example.get('mbert_bio_labels', []))

    if tokens and labels and len(tokens) == len(labels):
        mapped = [map_ai4privacy_label(x) for x in labels]
        return {
            'tokens': [str(t) for t in tokens],
            'ner_tags': mapped,
            'sequence': ' '.join(str(t) for t in tokens),
            'keep': True,
        }

    # Fallback fields for alternative dataset layouts.
    fallback_tokens = parse_list_field(example.get('tokens', []))
    fallback_labels = parse_list_field(example.get('ner_tags', example.get('labels', [])))
    if fallback_tokens and fallback_labels and len(fallback_tokens) == len(fallback_labels):
        mapped = [map_ai4privacy_label(x) for x in fallback_labels]
        return {
            'tokens': [str(t) for t in fallback_tokens],
            'ner_tags': mapped,
            'sequence': ' '.join(str(t) for t in fallback_tokens),
            'keep': True,
        }

    return {
        'tokens': [],
        'ner_tags': [],
        'sequence': '',
        'keep': False,
    }

# Load and prepare an English-only split for evaluation.
ai4 = load_dataset('ai4privacy/pii-masking-200k')
split_name = 'test' if 'test' in ai4 else 'train'
ai4_raw = ai4[split_name]

if 'language' in ai4_raw.column_names:
    ai4_raw = ai4_raw.filter(lambda x: str(x['language']).lower() == 'en')

ai4_conv = ai4_raw.map(convert_ai4privacy_example)
ai4_df = ai4_conv.to_pandas()
ai4_df = ai4_df[ai4_df['keep']].copy()
ai4_df = ai4_df[['tokens', 'ner_tags', 'sequence']]
ai4_df = ai4_df[ai4_df['tokens'].apply(len) > 0].reset_index(drop=True)

print('AI4Privacy usable rows:', len(ai4_df))
if len(ai4_df) == 0:
    raise ValueError('No valid AI4Privacy rows after conversion.')

# Build a small subset with PER/EMAIL presence so metrics are informative.
MAX_AI4_SAMPLES = 20
MIN_ROWS_WITH_PER = 5
MIN_ROWS_WITH_EMAIL = 5

def _row_has_entity(tags, suffix):
    return any(str(t).endswith(suffix) for t in tags)

ai4_df['has_per'] = ai4_df['ner_tags'].apply(lambda tags: _row_has_entity(tags, 'PER'))
ai4_df['has_email'] = ai4_df['ner_tags'].apply(lambda tags: _row_has_entity(tags, 'EMAIL'))

picked_idx = []
picked_set = set()

for idx in ai4_df[ai4_df['has_per']].index[:MIN_ROWS_WITH_PER]:
    if idx not in picked_set:
        picked_idx.append(idx)
        picked_set.add(idx)

for idx in ai4_df[ai4_df['has_email']].index[:MIN_ROWS_WITH_EMAIL]:
    if idx not in picked_set:
        picked_idx.append(idx)
        picked_set.add(idx)

for idx in ai4_df.index:
    if len(picked_idx) >= MAX_AI4_SAMPLES:
        break
    if idx not in picked_set:
        picked_idx.append(idx)
        picked_set.add(idx)

if not picked_idx:
    raise ValueError('Unable to construct AI4Privacy eval subset.')

ai4_eval = ai4_df.loc[picked_idx[:MAX_AI4_SAMPLES], ['tokens', 'ner_tags', 'sequence']].reset_index(drop=True)
ai4_per_tokens = sum(sum(1 for t in tags if str(t).endswith('PER')) for tags in ai4_eval['ner_tags'])
ai4_email_tokens = sum(sum(1 for t in tags if str(t).endswith('EMAIL')) for tags in ai4_eval['ner_tags'])
print(f'Selected AI4Privacy rows: {len(ai4_eval)} | PER tokens: {ai4_per_tokens} | EMAIL tokens: {ai4_email_tokens}')

ai4_true = []
ai4_pred = []
ai4_records = []

for idx, row in ai4_eval.iterrows():
    tokens = list(row['tokens'])
    true_tags = normalize_tag_list(row['ner_tags'])

    pred_tags, raw_response = run_zeroshot_token_tagger(tokens)

    ai4_true.extend(true_tags)
    ai4_pred.extend(pred_tags)

    # Capture a few sample rows for qualitative inspection.
    if idx < 5:
        ai4_records.append({
            'tokens': tokens[:20],
            'true_tags': true_tags[:20],
            'pred_tags': pred_tags[:20],
            'raw_response': raw_response,
        })

ai4_name = entity_binary_metrics(ai4_true, ai4_pred, 'PER')
ai4_email = entity_binary_metrics(ai4_true, ai4_pred, 'EMAIL')
ai4_report_df = pd.DataFrame([ai4_name, ai4_email], index=['NAME', 'EMAIL'])

display(ai4_report_df)
print(f'AI4Privacy rows evaluated: {len(ai4_eval)}')
print(f'AI4Privacy tokens evaluated: {len(ai4_true)}')

# Save external-evaluation metrics as a standalone report.
out_dir = BASE_DIR_OUTPUT / 'reports'
out_dir.mkdir(parents=True, exist_ok=True)
ai4_out_path = out_dir / 'llm_zero_shot_ai4privacy_report.csv'
ai4_report_df.to_csv(ai4_out_path)
print('Saved:', ai4_out_path)

pd.DataFrame(ai4_records)

AI4Privacy usable rows: 43501


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Selected AI4Privacy rows: 20 | PER tokens: 23 | EMAIL tokens: 47


Both `max_new_tokens` (=288) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=288) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=288) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=408) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

,accuracy,FPR,FNR,precision,recall,F1,support_positive
NAME,0.977692,0.002976,0.869565,0.5,0.130435,0.206897,23
EMAIL,0.954413,0.000000,1.000000,0.0,0.000000,0.000000,47


AI4Privacy rows evaluated: 20
AI4Privacy tokens evaluated: 1031
Saved: /kaggle/working/reports/llm_zero_shot_ai4privacy_report.csv


,tokens,true_tags,pred_tags,raw_response
0,"[Dear, Omer, ,, as, per, our, records, ,, your...","[O, B-PER, O, O, O, O, O, O, O, O, O, O, O, O,...","[O, B-PER, O, O, O, O, O, O, O, O, O, O, O, O,...","{""items"":[{""i"":0,""tag"":""O""},{""i"":1,""tag"":""B-PE..."
1,"[Kat, ##tie, could, you, pl, ##eas, ##e, share...","[B-PER, I-PER, O, O, O, O, O, O, O, O, O, O, O...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","{""items"":[{""i"":0,""tag"":""O""},{""i"":1,""tag"":""O""},..."
2,"[Carleton, ,, the, new, interactive, education...","[B-PER, O, O, O, O, O, O, O, O, O, O, O, O, O,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","{""items"":[{""i"":0,""tag"":""O""},{""i"":1,""tag"":""O""},..."
3,"[I, ', m, Ga, ##rri, ##ck, Murray, ., I, will,...","[O, O, O, B-PER, I-PER, I-PER, B-PER, O, O, O,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","{""items"":[{""i"":0,""tag"":""O""},{""i"":1,""tag"":""O""},..."
4,"["", Dear, Carmen, ,, As, our, National, Brand,...","[O, O, B-PER, O, O, O, O, O, O, O, O, O, O, O,...","[O, O, B-PER, I-PER, O, O, O, O, O, O, O, O, O...","{""items"":[{""i"":0,""tag"":""O""},{""i"":1,""tag"":""O""},..."
